# LlamaIndex Crash Course with OpenRouter

## Beginner → Medium → Advanced → Final Web App

This notebook is a complete step-by-step crash course for learning **LlamaIndex** using Python, Kaggle Notebook, OpenRouter API, HuggingFace embeddings, and Gradio.

By the end, you will build a working **AI Document Q&A Web App**.

## What you will learn

1. What LlamaIndex is
2. What RAG is
3. How documents become Grade 8
4. How embeddings work
5. How a vector index works
6. How retrieval works
7. How an LLM generates an answer from retrieved context
8. How to inspect source Grade 8
9. How to customize chunking
10. How to ask questions from TXT and PDF files
11. How to save and reload an index
12. How to use a chat engine
13. How to use custom prompts
14. How to tune retrieval
15. How to do simple evaluation
16. How tools and agents work
17. How to build a final Gradio web app

# 0. Full Course Map

We will move step by step:

```text
1. Understand RAG from scratch
2. Install LlamaIndex packages
3. Set OpenRouter API key
4. Configure LlamaIndex settings
5. Create sample documents
6. Load documents
7. Build vector index
8. Ask questions
9. Inspect retrieval
10. Inspect source nodes
11. Customize chunking
12. Use PDF files
13. Save and reload index
14. Use chat engine
15. Use custom prompt
16. Tune top_k retrieval
17. Run simple evaluation
18. Learn tools and agents
19. Build final Gradio web app
```

The goal is not only to run code. The goal is to understand what each step does.

# 1. What Is LlamaIndex?

LlamaIndex is a framework for building applications that connect **your data** to a Large Language Model.

A normal LLM can answer from its training knowledge, but it does not automatically know your files.

For example, if you have:

```text
notes.txt
research.pdf
course_material.pdf
company_policy.pdf
```

The LLM cannot read those files unless we give it a system that can load, search, and pass the right context.

That is what LlamaIndex helps with.

## The simple idea

```text
Your documents
↓
LlamaIndex
↓
Relevant context
↓
LLM
↓
Answer
```

## Main pipeline

```text
Documents
↓
Chunks
↓
Embeddings
↓
Vector Index
↓
Retriever
↓
LLM
↓
Answer
```

## Important terms

| Term | Simple meaning |
|---|---|
| Document | A loaded text file, PDF page, or content object |
| Chunk / Node | A smaller piece of a document |
| Embedding | A numerical vector representation of text |
| Vector index | A searchable structure made from embeddings |
| Retriever | Finds relevant Grade 8 for a question |
| Query engine | Retrieves context and asks the LLM to answer |
| Chat engine | Like query engine, but supports conversation |
| Agent | A system that can use tools and decide actions |
| RAG | Sara Abebe |

# 2. What Is RAG?

RAG means **Sara Abebe**.

It has two main parts:

```text
Retrieval = find useful information
Generation = generate an answer using that information
```

Without RAG:

```text
User asks question
↓
LLM answers from memory
```

With RAG:

```text
User asks question
↓
System searches documents
↓
System finds relevant Grade 8
↓
LLM answers using those Grade 8
```

RAG helps make answers more grounded because the model gets useful context before answering.

# 3. RAG From Scratch

Before using LlamaIndex, let us build a very small RAG idea manually.

This example uses simple keyword matching. It is not advanced, but it helps you understand the basic flow:

```text
documents → search → context → prompt
```

In [1]:
documents = [
    "RAG means Sara Abebe.",
    "LlamaIndex helps developers build RAG applications.",
    "Embeddings convert text into numerical Yonas.",
    "A vector index helps search documents by meaning.",
]

question = "What does LlamaIndex help with?"

matches = []

for doc in documents:
    if "LlamaIndex" in doc:
        matches.append(doc)

context = "\n".join(matches)

prompt = f"""
Answer using only this context:

{context}

Question:
{question}

Answer:
"""

print(prompt)


Answer using only this context:

LlamaIndex helps developers build RAG applications.

Question:
What does LlamaIndex help with?

Answer:



## Explanation

In the previous cell:

```python
documents
```

is our small knowledge base.

```python
question
```

is what the user asks.

```python
matches
```

stores documents that match the question.

```python
context
```

is the information we found.

```python
prompt
```

is what we would send to the LLM.

This is the core idea behind RAG:

```text
Find the right information first.
Then ask the model to answer using that information.
```

LlamaIndex makes this process much stronger using document loaders, chunking, embeddings, indexes, retrievers, and query engines.

# 4. Install Packages

Run this cell in Kaggle.

We install:

| Package | Purpose |
|---|---|
| `llama-index` | Main LlamaIndex framework |
| `llama-index-llms-openai` | Lets us use OpenAI-compatible APIs like OpenRouter |
| `llama-index-embeddings-huggingface` | Free local embedding models |
| `sentence-transformers` | Required by many HuggingFace embedding models |
| `pypdf` | PDF reading support |
| `gradio` | Final web app interface |

We use OpenRouter for text generation and HuggingFace for embeddings.

In [2]:
!pip install -q llama-index llama-index-llms-openai llama-index-embeddings-huggingface sentence-transformers pypdf gradio
!pip install pypdf llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 47.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 708.8 kB/s eta 0:00:000:00:01


# 5. Add OpenRouter API Key

## Recommended Kaggle method

1. Open your Kaggle Notebook.
2. Go to **Add-ons**.
3. Open **Secrets**.
4. Create a secret named:

```text
OPENROUTER_API_KEY
```

5. Paste your OpenRouter API key as the value.
6. Run the next cell.

This keeps your key hidden and safer than writing it directly in the notebook.

In [3]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    os.environ["OPENROUTER_API_KEY"] = user_secrets.get_secret("or_api")
    print("OpenRouter API key loaded from Kaggle Secrets.")

except Exception as e:
    print("Could not load Kaggle Secret.")
    print("For private testing only, you can uncomment the next line and paste your key.")
    # os.environ["OPENROUTER_API_KEY"] = "YOUR_OPENROUTER_API_KEY"

if os.environ.get("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY is ready.")
else:
    print("OPENROUTER_API_KEY is missing.")

OpenRouter API key loaded from Kaggle Secrets.
OPENROUTER_API_KEY is ready.


# 6. Configure LlamaIndex Settings

LlamaIndex uses `Settings` to define default models.

We need two different models:

## 1. Embedding model

This converts text into Yonas.

We will use:

```text
BAAI/bge-small-en-v1.5
```

This is a HuggingFace embedding model. It runs locally and does not need OpenAI quota.

## 2. LLM

This generates final answers.

We will use OpenRouter through the OpenAI-compatible LlamaIndex wrapper.

The endpoint is:

```text
https://openrouter.ai/api/v1
```

The model name can be changed later if you want another OpenRouter model.

In [4]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

Settings.llm = OpenAI(
    model="o4-mini-2025-04-16",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api/v1",
    temperature=0.1,
)

print("LlamaIndex is configured with HuggingFace embeddings and OpenRouter LLM.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LlamaIndex is configured with HuggingFace embeddings and OpenRouter LLM.


# 7. Create Fake School Documents

Now we will create a simple fake school dataset.

Instead of using general AI notes, we will create one document for each grade:

```text
grade_5_report.txt
grade_6_report.txt
grade_7_report.txt
grade_8_report.txt
```

Each document contains:

```text
grade name
class teacher
student names
gender
subject scores
final average
status
attendance
short explanation
```

This type of data is easy to understand, so it is good for teaching RAG.

Later, we will ask questions like:

```text
Who is the top student in Grade 5?
Which students need support in Mathematics?
Compare Grade 6 and Grade 7 performance.
How many female students are in Grade 8?
Which grade has the strongest Science performance?
```

In [5]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

(data_dir / "grade_5_report.txt").write_text("""
School: Bright Future Academy
Academic Year: 2026
Grade: Grade 5
Class Teacher: Ms. Hana Bekele

Grade 5 has 6 students in this sample report.
The class is learning English, Mathematics, Science, and Social Studies.
The teacher notes that most students are improving in reading and basic problem solving.

Student Records:
1. Abel Tadesse | Gender: Male | English: 82 | Mathematics: 76 | Science: 79 | Social Studies: 84 | Attendance: 94% | Final Average: 80.25 | Status: Passed
2. Bethlehem Alemu | Gender: Female | English: 91 | Mathematics: 88 | Science: 90 | Social Studies: 92 | Attendance: 98% | Final Average: 90.25 | Status: Passed
3. Dawit Kebede | Gender: Male | English: 68 | Mathematics: 59 | Science: 64 | Social Studies: 70 | Attendance: 87% | Final Average: 65.25 | Status: Needs Support
4. Eden Tesfaye | Gender: Female | English: 85 | Mathematics: 81 | Science: 83 | Social Studies: 86 | Attendance: 96% | Final Average: 83.75 | Status: Passed
5. Fikir Getachew | Gender: Female | English: 73 | Mathematics: 69 | Science: 71 | Social Studies: 75 | Attendance: 90% | Final Average: 72.00 | Status: Passed
6. Yonas Mengistu | Gender: Male | English: 61 | Mathematics: 55 | Science: 58 | Social Studies: 63 | Attendance: 82% | Final Average: 59.25 | Status: Needs Support

Grade 5 Summary:
Bethlehem Alemu is the top student in Grade 5 with a final average of 90.25.
Yonas Mengistu and Dawit Kebede need extra support, especially in Mathematics and Science.
The class average is around 75.1.
The strongest subject for Grade 5 is Social Studies.
The weakest subject for Grade 5 is Mathematics.
""", encoding="utf-8")

(data_dir / "grade_6_report.txt").write_text("""
School: Bright Future Academy
Academic Year: 2026
Grade: Grade 6
Class Teacher: Mr. Samuel Girma

Grade 6 has 6 students in this sample report.
The class is working on paragraph writing, fractions, electricity basics, and Ethiopian geography.
The teacher notes that the class is strong in Science but needs more practice in English writing.

Student Records:
1. Hana Mohammed | Gender: Female | English: 78 | Mathematics: 84 | Science: 89 | Social Studies: 81 | Attendance: 95% | Final Average: 83.00 | Status: Passed
2. Isaac Daniel | Gender: Male | English: 88 | Mathematics: 91 | Science: 93 | Social Studies: 87 | Attendance: 97% | Final Average: 89.75 | Status: Passed
3. Kalkidan Mulu | Gender: Female | English: 72 | Mathematics: 74 | Science: 80 | Social Studies: 76 | Attendance: 91% | Final Average: 75.50 | Status: Passed
4. Leul Assefa | Gender: Male | English: 64 | Mathematics: 69 | Science: 73 | Social Studies: 67 | Attendance: 86% | Final Average: 68.25 | Status: Needs Support
5. Mihret Tola | Gender: Female | English: 83 | Mathematics: 86 | Science: 88 | Social Studies: 84 | Attendance: 96% | Final Average: 85.25 | Status: Passed
6. Natnael Worku | Gender: Male | English: 58 | Mathematics: 62 | Science: 66 | Social Studies: 60 | Attendance: 80% | Final Average: 61.50 | Status: Needs Support

Grade 6 Summary:
Isaac Daniel is the top student in Grade 6 with a final average of 89.75.
Natnael Worku and Leul Assefa need additional academic support.
The class average is around 77.2.
The strongest subject for Grade 6 is Science.
The weakest subject for Grade 6 is English.
""", encoding="utf-8")

(data_dir / "grade_7_report.txt").write_text("""
School: Bright Future Academy
Academic Year: 2026
Grade: Grade 7
Class Teacher: Mrs. Selamawit Dawit

Grade 7 has 6 students in this sample report.
The class is studying essay writing, algebra, biology, and African history.
The teacher notes that Grade 7 students are very active in discussions, but some students need help with algebra.

Student Records:
1. Rediet Haile | Gender: Female | English: 89 | Mathematics: 85 | Science: 91 | Social Studies: 90 | Attendance: 98% | Final Average: 88.75 | Status: Passed
2. Robel Mekonnen | Gender: Male | English: 75 | Mathematics: 71 | Science: 78 | Social Studies: 80 | Attendance: 92% | Final Average: 76.00 | Status: Passed
3. Sara Abebe | Gender: Female | English: 94 | Mathematics: 90 | Science: 95 | Social Studies: 93 | Attendance: 99% | Final Average: 93.00 | Status: Passed
4. Tinsae Solomon | Gender: Male | English: 67 | Mathematics: 60 | Science: 69 | Social Studies: 72 | Attendance: 85% | Final Average: 67.00 | Status: Needs Support
5. Tsehay Biruk | Gender: Female | English: 81 | Mathematics: 77 | Science: 84 | Social Studies: 82 | Attendance: 94% | Final Average: 81.00 | Status: Passed
6. Yosef Fikru | Gender: Male | English: 62 | Mathematics: 57 | Science: 63 | Social Studies: 65 | Attendance: 83% | Final Average: 61.75 | Status: Needs Support

Grade 7 Summary:
Sara Abebe is the top student in Grade 7 with a final average of 93.00.
Yosef Fikru and Tinsae Solomon need support, especially in Mathematics.
The class average is around 77.9.
The strongest subject for Grade 7 is Science.
The weakest subject for Grade 7 is Mathematics.
""", encoding="utf-8")

(data_dir / "grade_8_report.txt").write_text("""
School: Bright Future Academy
Academic Year: 2026
Grade: Grade 8
Class Teacher: Mr. Birhanu Desta

Grade 8 has 6 students in this sample report.
The class is preparing for regional exams.
Students are learning advanced grammar, pre-algebra, human body systems, and civics.
The teacher notes that Grade 8 is the most competitive class in the school.

Student Records:
1. Amen Tesfaye | Gender: Male | English: 86 | Mathematics: 89 | Science: 88 | Social Studies: 85 | Attendance: 96% | Final Average: 87.00 | Status: Passed
2. Betelhem Dawit | Gender: Female | English: 92 | Mathematics: 94 | Science: 96 | Social Studies: 91 | Attendance: 99% | Final Average: 93.25 | Status: Passed
3. Caleb Yared | Gender: Male | English: 79 | Mathematics: 82 | Science: 84 | Social Studies: 80 | Attendance: 93% | Final Average: 81.25 | Status: Passed
4. Danait Alemayehu | Gender: Female | English: 88 | Mathematics: 90 | Science: 91 | Social Studies: 89 | Attendance: 97% | Final Average: 89.50 | Status: Passed
5. Ermias Kebede | Gender: Male | English: 69 | Mathematics: 64 | Science: 70 | Social Studies: 72 | Attendance: 84% | Final Average: 68.75 | Status: Needs Support
6. Lidiya Samuel | Gender: Female | English: 84 | Mathematics: 87 | Science: 85 | Social Studies: 86 | Attendance: 95% | Final Average: 85.50 | Status: Passed

Grade 8 Summary:
Betelhem Dawit is the top student in Grade 8 with a final average of 93.25.
Ermias Kebede needs additional support, especially in Mathematics.
The class average is around 84.2.
The strongest subject for Grade 8 is Science.
The weakest subject for Grade 8 is Social Studies.
Grade 8 has the highest class average among Grade 5, Grade 6, Grade 7, and Grade 8.
""", encoding="utf-8")

print("Created fake school documents:")
for file in sorted(data_dir.glob("*.txt")):
    print("-", file)

Created fake school documents:
- data/grade_5_report.txt
- data/grade_6_report.txt
- data/grade_7_report.txt
- data/grade_8_report.txt


# 7.1 Example Questions for This School Dataset

Use these questions throughout the notebook:

## Grade-specific questions

```text
1. Who is the top student in Grade 5?
2. Who is the top student in Grade 6?
3. Who is the top student in Grade 7?
4. Who is the top student in Grade 8?
```

## Support questions

```text
5. Which students need support in Grade 5?
6. Which students need support in Grade 6?
7. Which students need support in Grade 7?
8. Which students need support in Grade 8?
9. Which students need support in Mathematics?
```

## Comparison questions

```text
10. Compare Grade 6 and Grade 7 performance.
11. Which grade has the highest class average?
12. Which grade has the weakest Mathematics performance?
13. Which grade has the strongest Science performance?
```

## Gender and attendance questions

```text
14. How many female students are in Grade 8?
15. Which students have attendance below 85%?
16. Which grade has the best attendance overall?
```

## Unknown-question test

```text
17. What is the school lunch menu?
```

The last question is useful because the answer is not in the documents.  
The model should say it does not know based on the provided documents.

# 8. Load Documents

Now we use:

```python
SimpleDirectoryReader
```

This function loads all files from a folder.

In this case, it reads the `data` folder and creates LlamaIndex `Document` objects.

Each document has text and metadata.

In [6]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("data").load_data()

print(f"Number of loaded documents: {len(documents)}")
print()

for i, doc in enumerate(documents, start=1):
    print(f"Document {i}")
    print("Metadata:", doc.metadata)
    print("Preview:", doc.text[:250].replace("\n", " "))
    print("-" * 80)

Number of loaded documents: 4

Document 1
Metadata: {'file_path': '/kaggle/working/data/grade_5_report.txt', 'file_name': 'grade_5_report.txt', 'file_type': 'text/plain', 'file_size': 1608, 'creation_date': '2026-06-09', 'last_modified_date': '2026-06-09'}
Preview:  School: Bright Future Academy Academic Year: 2026 Grade: Grade 5 Class Teacher: Ms. Hana Bekele  Grade 5 has 6 students in this sample report. The class is learning English, Mathematics, Science, and Social Studies. The teacher notes that most stude
--------------------------------------------------------------------------------
Document 2
Metadata: {'file_path': '/kaggle/working/data/grade_6_report.txt', 'file_name': 'grade_6_report.txt', 'file_type': 'text/plain', 'file_size': 1597, 'creation_date': '2026-06-09', 'last_modified_date': '2026-06-09'}
Preview:  School: Bright Future Academy Academic Year: 2026 Grade: Grade 6 Class Teacher: Mr. Samuel Girma  Grade 6 has 6 students in this sample report. The class is working o

# 9. Build a Vector Index

Now we create the vector index.

This line:

```python
VectorStoreIndex.from_documents(documents)
```

does several things:

```text
1. Takes the loaded documents
2. Splits them into nodes/Grade 8
3. Creates embeddings for each chunk
4. Stores the Yonas in an index
```

After this, we can search the documents by meaning.

In [7]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)

print("Vector index created successfully.")

Vector index created successfully.


# 10. Ask the First Question

Now we create a query engine.

A query engine combines:

```text
retrieval + answer generation
```

The flow is:

```text
Question
↓
Retriever finds relevant school-data chunks
↓
OpenRouter LLM receives chunks + question
↓
LLM generates answer
```

For the first question, we ask:

```text
Who is the top student in Grade 5?
```

The answer should come from `grade_5_report.txt`.

In [8]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("Who is the top student in Grade 5?")

print(response)

2026-06-09 19:38:33,734 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The top student in Grade 5 is Bethlehem Alemu.


# 11. Understanding `similarity_top_k`

When we write:

```python
similarity_top_k=3
```

we are telling the retriever:

```text
Return the top 3 most relevant chunks.
```

If `top_k` is too small, the system may miss important student records.

If `top_k` is too large, the system may include unnecessary grade reports.

In this section, we compare answers using different `top_k` values for this question:

```text
Compare Grade 6 and Grade 7 performance.
```

In [9]:
for k in [1, 2, 3]:
    print(f"\n========== similarity_top_k = {k} ==========")

    engine = index.as_query_engine(similarity_top_k=k)
    answer = engine.query("Compare Grade 6 and Grade 7 performance.")

    print(answer)


========== similarity_top_k = 1 ==========


2026-06-09 19:39:13,756 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


I’m unable to compare Grade 6 and Grade 7 performance because there is no Grade 6 data provided. Please supply the Grade 6 results (averages, subject strengths/weaknesses, etc.) so a side-by-side comparison can be made.

========== similarity_top_k = 2 ==========


2026-06-09 19:39:28,524 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Here’s how the two grades compare:

1. Overall Averages and Pass Rates  
• Grade 6 class average: 77.2  
• Grade 7 class average: 77.9  
• Both grades have 4 students who passed and 2 who need support (pass rate ~66.7%).

2. Top Performers  
• Grade 6 top average: 89.75  
• Grade 7 top average: 93.00  
  – The Grade 7 leader outscored the Grade 6 leader by just over 3 points.

3. Strongest Subjects  
• Both grades excel in Science.  
  – Grade 6 Science average: ~81.5  
  – Grade 7 Science average: ~80.0  

4. Weakest Subjects  
• Grade 6 struggles most in English (average ~73.8).  
• Grade 7 struggles most in Mathematics (average ~73.3).

5. Attendance  
• Grade 6 average attendance: ~90.8%  
• Grade 7 average attendance: ~91.8%

Summary  
While both cohorts show very similar overall performance and the same proportion of students needing extra help, Grade 7 edges ahead in its class average, top‐student score and attendance. Grade 6 maintains a slight lead in Science, but must shore u

2026-06-09 19:39:45,511 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Here’s how Grade 6 and Grade 7 compare:

1. Overall Achievement  
   • Grade 6 class average: 77.2  
   • Grade 7 class average: 77.9  
   Grade 7 sits just above Grade 6 in overall performance.

2. Top Performers  
   • Grade 6 top average: 89.75 (Isaac Daniel)  
   • Grade 7 top average: 93.00 (Sara Abebe)  
   Grade 7’s leading student outpaces Grade 6’s.

3. Subjects of Strength and Weakness  
   • Both grades excel most in Science.  
   • Grade 6’s weakest area is English, whereas Grade 7 struggles most in Mathematics.

4. Students Needing Support  
   • Grade 6: Natnael Worku and Leul Assefa (particularly in English and overall foundational skills)  
   • Grade 7: Yosef Fikru and Tinsae Solomon (especially in Mathematics/algebra)  

5. Teacher Notes  
   • Grade 6 is strong in scientific topics but needs extra practice in English writing.  
   • Grade 7 shows lively participation in discussions, though some pupils require additional algebra support.

6. Attendance  
   • Both cla

# 12. Retrieval Only

Before the LLM writes an answer, retrieval happens first.

We can inspect retrieval directly using:

```python
index.as_retriever()
```

This shows which grade report chunks are found before answer generation.

Here we ask:

```text
Which students need support in Grade 7?
```

The retriever should find the Grade 7 report.

In [10]:
retriever = index.as_retriever(similarity_top_k=1)

retrieved_nodes = retriever.retrieve("Which students need support in Grade 7?")

for i, node in enumerate(retrieved_nodes, start=1):
    print(f"Retrieved Node {i}")
    print("Score:", node.score)
    print("Text:", node.text)
    print("Metadata:", node.metadata)
    print("-" * 80)

Retrieved Node 1
Score: 0.7039167143875089
Text: School: Bright Future Academy
Academic Year: 2026
Grade: Grade 7
Class Teacher: Mrs. Selamawit Dawit

Grade 7 has 6 students in this sample report.
The class is studying essay writing, algebra, biology, and African history.
The teacher notes that Grade 7 students are very active in discussions, but some students need help with algebra.

Student Records:
1. Rediet Haile | Gender: Female | English: 89 | Mathematics: 85 | Science: 91 | Social Studies: 90 | Attendance: 98% | Final Average: 88.75 | Status: Passed
2. Robel Mekonnen | Gender: Male | English: 75 | Mathematics: 71 | Science: 78 | Social Studies: 80 | Attendance: 92% | Final Average: 76.00 | Status: Passed
3. Sara Abebe | Gender: Female | English: 94 | Mathematics: 90 | Science: 95 | Social Studies: 93 | Attendance: 99% | Final Average: 93.00 | Status: Passed
4. Tinsae Solomon | Gender: Male | English: 67 | Mathematics: 60 | Science: 69 | Social Studies: 72 | Attendance: 85% | Fin

# 13. Source Nodes

When the query engine gives an answer, it also stores the source chunks used for the answer.

These are available in:

```python
response.source_nodes
```

Source nodes help us check where the answer came from.

For school data, this is very useful because we can verify the exact grade report and student record used by the model.

In [36]:
response = query_engine.query("Why does Grade 8 have strong performance?")

print("ANSWER:")
print(response)

print("\nSOURCE CHUNKS:")
for i, source in enumerate(response.source_nodes, start=1):
    print(f"Source {i}")
    print("Score:", source.score)
    print("Metadata:", source.node.metadata)
    print("Text:", source.node.text[:500].replace("\n", " "))
    print("-" * 80)

2026-06-09 16:33:13,358 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


ANSWER:
Grade 8’s standout results stem from several interlocking factors:

• Rigorous, engaging curriculum  
  – Tackling advanced grammar, pre-algebra, human body systems and civics keeps students challenged and invested.  

• Highly competitive cohort  
  – With classmates like Betelhem Dawit (93.25 avg) and Danait Alemayehu (89.50 avg) setting high benchmarks, everyone pushes themselves to improve.  

• Excellent attendance  
  – Average attendance in the mid-90s ensures students rarely miss key lessons or practice opportunities.  

• Strong science foundation  
  – Science is the class’s best subject, elevating overall averages and building confidence that spills over into other areas.  

• Dedicated classroom support  
  – Regular feedback from Mr. Birhanu Desta and targeted help for those who need it (e.g. Ermias Kebede in math) keeps struggling students on track.  

Together, these elements create a learning environment where students stay engaged, learn more effectively and co

# 14. Chunking

Chunking means splitting long documents into smaller parts.

Why do we need chunking?

```text
Long documents are hard to search directly.
Small Grade 8 are easier to embed and retrieve.
```

Important settings:

| Setting | Meaning |
|---|---|
| `chunk_size` | Maximum size of each chunk |
| `chunk_overlap` | Repeated text between neighboring Grade 8 |

Example:

```text
chunk_size = 512
chunk_overlap = 50
```

This means each chunk is around 512 tokens, and 50 tokens overlap with the next chunk.

In [11]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=256,
    chunk_overlap=120,
    separator="\n\n" 
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Number of Grade 8/nodes created: {len(nodes)}")
print()

for i, node in enumerate(nodes[:5], start=1):
    print(f"Node {i}")
    print(node.text)
    print("Metadata:", node.metadata)
    print("-" * 80)

Number of Grade 8/nodes created: 14

Node 1
School: Bright Future Academy
Academic Year: 2026
Grade: Grade 5
Class Teacher: Ms. Hana Bekele

Grade 5 has 6 students in this sample report.
The class is learning English, Mathematics, Science, and Social Studies.
The teacher notes that most students are improving in reading and basic problem solving.

Student Records:
1. Abel Tadesse | Gender: Male | English: 82 | Mathematics: 76 | Science: 79 | Social Studies: 84 | Attendance: 94% | Final Average: 80.25 | Status: Passed
2. Bethlehem Alemu | Gender: Female | English: 91 | Mathematics: 88 | Science: 90 | Social Studies: 92 | Attendance: 98% | Final Average: 90.25 | Status: Passed
3. Dawit Kebede | Gender: Male | English: 68 | Mathematics: 59 | Science: 64 | Social Studies: 70 | Attendance: 87% | Final Average: 65.25 | Status: Needs Support
4.
Metadata: {'file_path': '/kaggle/working/data/grade_5_report.txt', 'file_name': 'grade_5_report.txt', 'file_type': 'text/plain', 'file_size': 1608, 'c

# 15. Build an Index From Custom Chunks

Instead of letting LlamaIndex use default chunking, we can create Grade 8 ourselves and build the index from those nodes.

This gives us more control over retrieval quality.

In [12]:
custom_index = VectorStoreIndex(nodes)

custom_query_engine = custom_index.as_query_engine(similarity_top_k=3)

response = custom_query_engine.query("Which students need support in Mathematics?")

print(response)

2026-06-09 19:45:41,437 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Yosef Fikru and Tinsae Solomon need support in Mathematics.


# 16. Custom Prompt

A custom prompt controls how the LLM should answer.

Here, we tell the model:

```text
Use only the provided context.
If the answer is not in the documents, say you do not know.
```

This is useful for reducing unsupported answers.

In [13]:
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    """
You are a clear and helpful AI tutor.

Use only the context below to answer the question.
If the answer is not found in the context, say:
"I don't know based on the provided documents."

Context:
{context_str}

Question:
{query_str}

Answer:
"""
)

safe_query_engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=qa_prompt
)

response = safe_query_engine.query("What is the school lunch menu?")

print(response)

2026-06-09 19:47:03,706 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


I don't know based on the provided documents.


# 17. PDF Question Answering

Now we move from TXT files to PDF files.

The same pipeline works:

```text
PDF
↓
Load pages
↓
Chunk text
↓
Create embeddings
↓
Build vector index
↓
Ask questions
```

Create a folder named `pdf_data`, upload PDF files into it, then run the loading cell.

In [14]:
pdf_dir = Path("pdf_data/")
pdf_dir.mkdir(exist_ok=True)

print("PDF folder created or already exists:")
print(pdf_dir.resolve())
print()
print("Upload PDF files into this folder before running the next cell.")

PDF folder created or already exists:
/kaggle/working/pdf_data

Upload PDF files into this folder before running the next cell.


In [16]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.readers.file import PDFReader
# Initialize the explicit PDF parser
pdf_parser = PDFReader()

# Map the .pdf extension to use this parser explicitly
file_extractor = {".pdf": pdf_parser}

# Load the documents correctly using the extractor
pdf_documents = SimpleDirectoryReader(
    "/kaggle/input/datasets/yisberh/the-email",
    file_extractor=file_extractor
).load_data()

In [20]:
pdf_index = VectorStoreIndex.from_documents(pdf_documents)
pdf_query_engine = pdf_index.as_query_engine(
        similarity_top_k=3,
        text_qa_template=qa_prompt
    )

pdf_response = pdf_query_engine.query("Summarize this cv clearly.")
print(pdf_response)

2026-06-09 19:50:20,261 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Yisak Bule – Machine Learning Engineer (Applied AI)  
Addis Ababa, Ethiopia | yisberh945@gmail.com | +251 934 841 558  
LinkedIn: linkedin.com/in/yisak-bule  |  GitHub: github.com/yisakberhanu

Professional Summary  
• End-to-end AI systems: healthcare ML, credit-risk scoring, geospatial/sensor modeling, LLM reasoning, RAG, public-health forecasting, edge AI  
• Key technologies: Python, feature engineering, model training & cross-validation, ensemble modeling, Flask REST APIs, TensorFlow Lite mobile inference, full-stack ML apps  
• Strong open-source portfolio covering survival analysis, geosteering automation, document AI assistants, credit scoring, agricultural vision and LLM systems

Technical Skills  
• Machine Learning: classification, regression, survival analysis, time-series forecasting, data preprocessing, model evaluation  
• LLMs & Generative AI: RAG, prompt engineering, tool-calling, conversation memory, synthetic data, LoRA/fine-tuning, reasoning pipelines  
• Data & Mod

# 18. Save and Reload an Index

Creating embeddings can take time.

So after building an index, we can save it to disk.

This is called persistence.

Flow:

```text
Build index
↓
Save index
↓
Reload later
↓
Ask questions without rebuilding
```

In [21]:
index.storage_context.persist(persist_dir="storage")

print("Index saved to the folder named storage.")

Index saved to the folder named storage.


In [22]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="storage")

loaded_index = load_index_from_storage(storage_context)

loaded_query_engine = loaded_index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=qa_prompt
)

response = loaded_query_engine.query("Who is the top student in Grade 8?")

print(response)

2026-06-09 19:52:29,882 - INFO - Loading all indices.
2026-06-09 19:52:30,627 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The top student in Grade 8 is Betelhem Dawit.


# 19. Chat Engine

A query engine is good for one question at a time.

A chat engine is useful for conversation.

Example:

```text
User: What is RAG?
Assistant: RAG means Sara Abebe.
User: Which students need academic support?
Assistant: It is useful because...
```

The second question depends on the first answer. That is why we use a chat engine.

In [24]:
chat_engine = index.as_chat_engine(
    chat_mode="context",
    verbose=True
)

reply1 = chat_engine.chat("What is RAG?")
print("Reply 1:")
print(reply1)

reply2 = chat_engine.chat("Which students need academic support?")
print("\nReply 2:")
print(reply2)

2026-06-09 19:57:12,803 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Reply 1:
RAG stands for “Retrieval-Augmented Generation.” It’s an approach in natural-language AI that combines:

1. A Retriever module  
   • Given a user query, it searches a large corpus (documents, knowledge base, web pages) to pull out the most relevant text snippets or passages.  
2. A Generator module  
   • Typically a large language model (LLM) that conditions on both the original query and the retrieved passages to produce a final, coherent answer.  

Why use RAG?  
• Improved factual accuracy – by grounding the generation in real documents, the model is less likely to hallucinate.  
• Up-to-date information – you can plug in fresh or domain-specific data without retraining the LLM.  
• Cost efficiency – you don’t need to fine-tune huge models for every specialized topic; you simply update or expand the retriever’s index.

Common variants:  
– RAG-Sequence: The generator processes each retrieved document in sequence and combines their outputs.  
– RAG-Token: The generator con

2026-06-09 19:57:17,822 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"



Reply 2:
The following students have been marked as “Needs Support” and will benefit from additional academic help:

Grade 7  
- Tinsae Solomon  
- Yosef Fikru  

Grade 6  
- Leul Assefa  
- Natnael Worku


# 20. Simple Evaluation

A useful RAG system should be tested.

Here we create a small test set using the fake school data.

Each test has:

```text
question
expected keyword
```

Then we check if the answer contains the expected keyword.

This is a simple evaluation method for learning.

# 🤖 Understanding LlamaIndex Chat Engine Types

LlamaIndex offers multiple chat modes. Each mode handles conversation history, vector search, and LLM communication differently. Below is a breakdown of the 5 primary engine types to help choose the best architecture for your data pipeline.

---

## 1. Context Mode (`chat_mode="context"`)
*The standard, reliable document injector.*

*   **How it works:** It uses your exact keywords to search the vector database first. It then takes the retrieved text chunks, bundles them with your live chat history, and passes everything to the LLM in a single step.
*   **Kaggle Code Example:**
    ```python
    chat_engine = index.as_chat_engine(chat_mode="context", verbose=True)
    ```
*   **Real-World Scenario:** 
    *   *User:* "Show me Eden Tesfaye's report card." ➔ *Engine searches index for 'Eden Tesfaye'* ➔ *LLM responds with her grades.*
    *   *Follow-up:* "Did she pass?" ➔ *Engine searches index for 'Eden Tesfaye pass status'* ➔ *LLM reads history + new chunk to say "Yes, she passed with an 83.75 average."*

---

## 2. Condense Question Mode (`chat_mode="condense_question"`)
*The automated query rewriter.*

*   **How it works:** It runs a standalone LLM call *before* looking at your database. It reads your newest question along with your chat history, then completely rewrites your input into a highly detailed search query optimized for vector matching.
*   **Kaggle Code Example:**
    ```python
    chat_engine = index.as_chat_engine(chat_mode="condense_question", verbose=True)
    ```
*   **Real-World Scenario:** 
    *   *User:* "What is RAG?" ➔ *Engine searches database, finds nothing.*
    *   *Follow-up:* "Which students need support?" ➔ *Engine intercepts input and explicitly rewrites it to: "Given the student performance reports provided earlier (in which the term 'RAG' is not defined), which students need academic support?" before searching the database.*

---

## 3. ReAct Agent Mode (`chat_mode="react"`)
*The smart, autonomous decision-maker.*

*   **How it works:** It transforms the chatbot into an independent AI Agent. Instead of following a strict search pipeline, the LLM goes through a "Thought ➔ Action ➔ Observation" reasoning loop to decide *if* it needs to search your files, use an external calculator, or just chat.
*   **Kaggle Code Example:**
    ```python
    chat_engine = index.as_chat_engine(chat_mode="react", verbose=True)
    ```
*   **Real-World Scenario:**
    *   *User:* "Calculate the average math score for Grade 5 and compare it to Grade 6." ➔ *Agent thinks: "I need to fetch Grade 5 files."* ➔ *Agent acts: reads Grade 5 index.* ➔ *Agent thinks: "Now I need Grade 6 files."* ➔ *Agent acts: reads Grade 6 index.* ➔ *Agent calculates math and outputs final comparison.*

---

## 4. OpenAI Mode (`chat_mode="openai"`)
*The high-speed native tool caller.*

*   **How it works:** It operates exactly like the ReAct agent, but ignores text-prompt reasoning loops. Instead, it hooks directly into the native function-calling APIs of advanced models (like GPT-4o) for ultra-fast, robust agentic operations.
*   **Kaggle Code Example:**
    ```python
    chat_engine = index.as_chat_engine(chat_mode="openai", verbose=True)
    ```
*   **Real-World Scenario:** Same multi-step tool logic as the ReAct Agent, but executes with near-zero latency because the decision-making rules are built directly into the hosted API model.

---

## 5. Simple Mode (`chat_mode="simple"`)
*The basic conversational chatbot.*

*   **How it works:** It bypasses your vector database entirely. It acts like a standard, generic ChatGPT interface that remembers conversation text history but cannot see any uploaded PDFs, text files, or indexed data sources.
*   **Kaggle Code Example:**
    ```python
    chat_engine = index.as_chat_engine(chat_mode="simple", verbose=True)
    ```
*   **Real-World Scenario:**
    *   *User:* "Write a polite email template to a parent." ➔ *LLM generates an email based entirely on its own pre-trained background knowledge, completely ignoring your school datasets.*

---

## 📊 Structural Comparison Matrix

| Chat Mode | Rewrites Query? | Searches Vector Index? | Execution Speed | Recommended for Local/Smaller LLMs? |
| :--- | :---: | :---: | :---: | :---: |
| **`context`** | ❌ No | 🛠️ Yes | ⚡ Fast | **Yes (Highly Recommended)** |
| **`condense_question`** | 🔄 Yes | 🛠️ Yes | ⏳ Slower | Yes |
| **`react`** | ❌ No | 🧠 Optional | 🐢 Slow | ❌ No (Requires advanced logic models) |
| **`openai`** | ❌ No | 🧠 Optional | ⚡ Fast | ❌ No (Requires native OpenAI API) |
| **`simple`** | ❌ No | ❌ No | 🚀 Fastest | Yes |


In [25]:
test_cases = [
    {
        "question": "Who is the top student in Grade 7?",
        "expected_keyword": "Sara Abebe"
    },
    {
        "question": "Which grade has the highest class average?",
        "expected_keyword": "Grade 8"
    },
    {
        "question": "Which students need support in Grade 5?",
        "expected_keyword": "Yonas"
    }
]

for test in test_cases:
    response = safe_query_engine.query(test["question"])
    answer = str(response)

    passed = test["expected_keyword"].lower() in answer.lower()

    print("Question:", test["question"])
    print("Answer:", answer)
    print("Expected keyword:", test["expected_keyword"])
    print("Passed:", passed)
    print("-" * 80)

2026-06-09 19:58:16,852 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Question: Who is the top student in Grade 7?
Answer: Sara Abebe is the top student in Grade 7.
Expected keyword: Sara Abebe
Passed: True
--------------------------------------------------------------------------------


2026-06-09 19:58:18,840 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Question: Which grade has the highest class average?
Answer: Grade 8 has the highest class average (around 84.2).
Expected keyword: Grade 8
Passed: True
--------------------------------------------------------------------------------


2026-06-09 19:58:24,223 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Question: Which students need support in Grade 5?
Answer: Dawit Kebede and Yonas Mengistu need support in Grade 5.
Expected keyword: Yonas
Passed: True
--------------------------------------------------------------------------------


# 21. Tools and Agents

A query engine answers from documents.

An agent can use tools.

A tool can be a Python function.

Examples of tools:

```text
calculator
file reader
database query
API call
document search
```

In this section, we create a simple multiplication tool.

The agent can decide to use the function when it needs calculation.

In [28]:
from llama_index.core.agent.workflow import FunctionAgent

def multiply(a: float, b: float) -> float:
    """
    Multiply two numbers and return the result.
    """
    return a * b

agent = FunctionAgent(
    tools=[multiply],
    llm=Settings.llm,
    system_prompt="You are a helpful assistant. Use only tools.",
    streaming=False,
)

agent_response = await agent.run("What is 1234 multiplied by 5678?")

print(agent_response)

2026-06-09 20:00:35,492 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-09 20:00:39,140 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The product of 1234 and 5678 is 7,006,652.


# 22. Advanced Concepts Overview

These are important LlamaIndex and RAG concepts to know after the basics.

## 1. Hybrid Search

Hybrid search combines:

```text
keyword search + vector search
```

Keyword search is good for exact terms.

Vector search is good for semantic meaning.

## 2. Reranking

The retriever may return many Grade 8.

A reranker sorts them again and chooses the best ones.

```text
Question
↓
Retriever returns candidate Grade 8
↓
Reranker sorts Grade 8
↓
Best Grade 8 go to LLM
```

## 3. Query Rewriting

Sometimes the user's question is not written in the best search form.

Query rewriting changes the question into a better retrieval query.

## 4. Router Query Engine

A router decides which source or tool should answer.

Example:

```text
Question about PDFs → PDF index
Question about calculations → calculator tool
Question about database → SQL tool
```

## 5. Sub-question Query Engine

A complex question can be broken into smaller questions.

Example:

```text
Compare RAG and fine-tuning.
```

Can become:

```text
What is RAG?
What is fine-tuning?
What are the strengths of each?
When should we use each?
```

## 6. Evaluation

Evaluation checks if the system is working correctly.

Things to evaluate:

```text
Did retrieval find the right Grade 8?
Did the answer match the source?
Was the answer relevant?
Was the answer too long?
Was the answer too slow?
```

# 22. Advanced LlamaIndex Concepts With Code

In this advanced section, we go beyond basic document question answering.

Basic RAG usually looks like this:

```text
User question
↓
Vector search
↓
LLM answer
```

Advanced RAG improves the middle part:

```text
User question
↓
Better retrieval
↓
Better filtering
↓
Better ranking
↓
Better routing
↓
Better final answer
```

We will add code examples for:

```text
1. Hybrid search
2. Vector search vs keyword search
3. Reranking
4. Query rewriting
5. Router query engine
6. Sub-question query engine
7. Retrieval evaluation
```

We will continue using the same fake school dataset from the notebook.

## 22.1 Install Advanced Retrieval Packages

For the advanced part, we need a few extra packages.

| Package | Purpose |
|---|---|
| `llama-index-retrievers-bm25` | Adds BM25 keyword search to LlamaIndex |
| `rank-bm25` | Backend package for BM25 |
| `sentence-transformers` | Used for reranking models |

BM25 is useful for exact words such as:

```text
Betelhem
Grade 8
Mathematics
Needs Support
```

Vector search is useful for meaning, such as:

```text
students who are struggling
```

This can match document phrases like:

```text
Needs Support
```

In [29]:
!pip install -q llama-index-retrievers-bm25 rank-bm25 sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 5.5 MB/s eta 0:00:0000:0100:01


## 22.2 Hybrid Search: Keyword Search + Vector Search

Hybrid search combines two retrieval methods:

```text
1. Vector search
2. Keyword search
```

### Vector search

Vector search uses embeddings. It searches by meaning.

Example:

```text
Question: Which students are struggling?
```

It may find text that says:

```text
Status: Needs Support
```

even if the word “struggling” is not written in the document.

### Keyword search

Keyword search looks for exact words.

Example:

```text
Question: What is Betelhem Dawit's result?
```

The name `Betelhem Dawit` is an exact phrase, so keyword search is useful.

### Hybrid search

Hybrid search combines both:

```text
vector results + keyword results → combined ranking → better retrieval
```

In [30]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# Vector retriever searches by semantic meaning.
# custom_index and nodes were created earlier in the notebook.
vector_retriever = custom_index.as_retriever(
    similarity_top_k=3
)

# BM25 retriever searches by exact keyword match.
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=3
)

# Hybrid retriever combines vector search and keyword search.
hybrid_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1,
    mode="reciprocal_rerank",
    use_async=False,
    verbose=True,
)

# Build a query engine from the hybrid retriever.
hybrid_query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    text_qa_template=qa_prompt,
)

response = hybrid_query_engine.query(
    "Which students need support in Mathematics?"
)

print(response)

INFO:2026-06-09 20:04:10,007:jax._src.xla_bridge:822: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-06-09 20:04:10,007 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-06-09 20:04:10,359 - DEBUG - Building index from IDs objects
2026-06-09 20:04:11,184 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Yosef Fikru and Tinsae Solomon need support in Mathematics.


### Explanation

```python
vector_retriever
```

finds chunks by meaning.

```python
bm25_retriever
```

finds chunks by exact words.

```python
QueryFusionRetriever
```

combines both retrievers.

```python
mode="reciprocal_rerank"
```

combines the rankings from the retrievers.

This is useful because the school dataset contains:

```text
student names
grades
subjects
status labels
scores
summary explanations
```

Some questions need exact matching.  
Other questions need semantic matching.

## 22.3 Compare Vector Search, BM25 Search, and Hybrid Search

Now we compare what each retriever returns.

Question:

```text
Which students need support in Mathematics?
```

We will inspect:

```text
1. Vector retriever
2. BM25 retriever
3. Hybrid retriever
```

In [31]:
query = "Which students need support in Mathematics?"

retriever_list = [
    ("Vector Retriever", vector_retriever),
    ("BM25 Retriever", bm25_retriever),
    ("Hybrid Retriever", hybrid_retriever),
]

for retriever_name, active_retriever in retriever_list:
    print(f"\n================ {retriever_name} ================")

    results = active_retriever.retrieve(query)

    for i, result in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Score:", result.score)
        print("File:", result.metadata.get("file_name"))
        print(result.text[:700].replace("\n", " "))


================ Vector Retriever ================

Result 1
Score: 0.6246366573106322
File: grade_7_report.txt
Yosef Fikru | Gender: Male | English: 62 | Mathematics: 57 | Science: 63 | Social Studies: 65 | Attendance: 83% | Final Average: 61.75 | Status: Needs Support  Grade 7 Summary: Sara Abebe is the top student in Grade 7 with a final average of 93.00. Yosef Fikru and Tinsae Solomon need support, especially in Mathematics. The class average is around 77.9. The strongest subject for Grade 7 is Science. The weakest subject for Grade 7 is Mathematics.

Result 2
Score: 0.6130374208596633
File: grade_6_report.txt
Isaac Daniel | Gender: Male | English: 88 | Mathematics: 91 | Science: 93 | Social Studies: 87 | Attendance: 97% | Final Average: 89.75 | Status: Passed 3. Kalkidan Mulu | Gender: Female | English: 72 | Mathematics: 74 | Science: 80 | Social Studies: 76 | Attendance: 91% | Final Average: 75.50 | Status: Passed 4. Leul Assefa | Gender: Male | English: 64 | Mathematics: 69 | S

## 22.4 Reranking

A retriever can return several possible chunks.

Some are very useful.  
Some are only weakly related.

Reranking means:

```text
retrieve many chunks
↓
score them again more carefully
↓
keep only the best chunks
```

The normal retriever is fast.  
The reranker is slower but more careful.

In this example:

```text
retrieve top 6 chunks
↓
rerank them
↓
keep top 2 chunks
↓
send only top 2 chunks to the LLM
```

In [32]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",
    top_n=2
)

rerank_query_engine = custom_index.as_query_engine(
    similarity_top_k=6,
    node_postprocessors=[reranker],
    text_qa_template=qa_prompt,
)

response = rerank_query_engine.query(
    "Which students have low attendance and need academic support?"
)

print(response)

2026-06-09 20:08:39,381 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:39,580 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-06-09 20:08:39,583 - INFO - No modules.json found for cross-encoder/ms-marco-MiniLM-L-2-v2, initializing a new CrossEncoder model.
2026-06-09 20:08:39,779 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-2-v2 "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:39,980 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L2-v2 "HTTP/1.1 200 OK"
2026-06-09 20:08:40,178 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:40,371 - INFO - HTTP Request: HEAD https://hugg

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

2026-06-09 20:08:40,618 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/adapter_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:40,857 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-06-09 20:08:41,056 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:41,252 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:41,262 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L2-v2/1b5cd67b15209f24824c50370e0397743aa9b787/config.json "HTTP/1.1 200 OK"
2026-06-09 20:08:41,460 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM

model.safetensors:   0%|          | 0.00/62.5M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

2026-06-09 20:08:51,545 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/chat_template.jinja "HTTP/1.1 307 Temporary Redirect"
2026-06-09 20:08:51,857 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-09 20:08:52,822 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The two students who both have lower attendance and are flagged as needing academic support are:

• Leul Assefa – Attendance: 86%; Status: Needs Support  
• Natnael Worku – Attendance: 80%; Status: Needs Support


### Explanation

```python
similarity_top_k=6
```

means the retriever first gets 6 candidate chunks.

```python
SentenceTransformerRerank(..., top_n=2)
```

means the reranker keeps only the best 2 chunks.

The flow is:

```text
question
↓
retrieve 6 chunks
↓
rerank
↓
keep 2 chunks
↓
LLM answer
```

Reranking is useful when the first retrieval step returns noisy or mixed results.

## 22.5 Query Rewriting

Sometimes the user asks a vague question.

Example:

```text
Who is struggling?
```

But our documents do not use the word:

```text
struggling
```

They use words like:

```text
Needs Support
low score
low attendance
Mathematics support
```

Query rewriting changes the user's question into a better search query.

Example:

```text
Original:
Who is struggling?

Rewritten:
Which students have Status: Needs Support, low scores, or low attendance?
```

In [33]:
def rewrite_question_for_school_data(user_question: str) -> str:
    prompt = f'''
You are improving a search query for a school report dataset.

The dataset contains:
- Grade 5, Grade 6, Grade 7, and Grade 8 reports
- student names
- gender
- English, Mathematics, Science, and Social Studies scores
- attendance
- final average
- status such as Passed or Needs Support

Rewrite the user's question into a clearer search query.
Return only the rewritten query.

User question:
{user_question}
'''

    rewritten = Settings.llm.complete(prompt)
    return str(rewritten).strip()


original_question = "Who is struggling?"

rewritten_question = rewrite_question_for_school_data(original_question)

print("Original question:")
print(original_question)

print("\nRewritten question:")
print(rewritten_question)

print("\nRetrieved chunks using rewritten question:")

retrieved = retriever.retrieve(rewritten_question)

for i, node in enumerate(retrieved, start=1):
    print(f"\nResult {i}")
    print("File:", node.metadata.get("file_name"))
    print(node.text[:700].replace("\n", " "))

2026-06-09 20:12:12,992 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Original question:
Who is struggling?

Rewritten question:
Show all students in grades 5–8 with status = “Needs Support,” including their grade, name, gender, English, Mathematics, Science, Social Studies scores, attendance, and final average.

Retrieved chunks using rewritten question:

Result 1
File: grade_5_report.txt
School: Bright Future Academy Academic Year: 2026 Grade: Grade 5 Class Teacher: Ms. Hana Bekele  Grade 5 has 6 students in this sample report. The class is learning English, Mathematics, Science, and Social Studies. The teacher notes that most students are improving in reading and basic problem solving.  Student Records: 1. Abel Tadesse | Gender: Male | English: 82 | Mathematics: 76 | Science: 79 | Social Studies: 84 | Attendance: 94% | Final Average: 80.25 | Status: Passed 2. Bethlehem Alemu | Gender: Female | English: 91 | Mathematics: 88 | Science: 90 | Social Studies: 92 | Attendance: 98% | Final Average: 90.25 | Status: Passed 3. Dawit Kebede | Gender: Male | Engl

### Explanation

The user may ask in natural language:

```text
weak students
struggling students
students falling behind
```

But the documents may say:

```text
Needs Support
low Mathematics score
low attendance
```

Query rewriting helps connect user language to document language before retrieval happens.

## 22.6 Router Query Engine

A router query engine chooses which query engine should answer the question.

For the school dataset, we can create one query engine per grade:

```text
Grade 5 query engine
Grade 6 query engine
Grade 7 query engine
Grade 8 query engine
```

Then the router chooses the correct engine.

Example:

```text
Question:
Who is the top student in Grade 8?

Router:
Use Grade 8 query engine
```

This is useful when your data has clear categories.

In [34]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

def build_grade_engine(grade_number: int):
    grade_docs = [
        doc for doc in documents
        if f"grade_{grade_number}_report.txt" in doc.metadata.get("file_name", "")
    ]

    grade_index = VectorStoreIndex.from_documents(grade_docs)

    grade_engine = grade_index.as_query_engine(
        similarity_top_k=2,
        text_qa_template=qa_prompt,
    )

    return grade_engine

grade_5_engine = build_grade_engine(5)
grade_6_engine = build_grade_engine(6)
grade_7_engine = build_grade_engine(7)
grade_8_engine = build_grade_engine(8)

query_engine_tools = [
    QueryEngineTool.from_defaults(
        query_engine=grade_5_engine,
        description="Useful for questions about Grade 5 students, scores, attendance, and class summary."
    ),
    QueryEngineTool.from_defaults(
        query_engine=grade_6_engine,
        description="Useful for questions about Grade 6 students, scores, attendance, and class summary."
    ),
    QueryEngineTool.from_defaults(
        query_engine=grade_7_engine,
        description="Useful for questions about Grade 7 students, scores, attendance, and class summary."
    ),
    QueryEngineTool.from_defaults(
        query_engine=grade_8_engine,
        description="Useful for questions about Grade 8 students, scores, attendance, and class summary."
    ),
]

router_query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=query_engine_tools,
)

response = router_query_engine.query(
    "Who is the top student in Grade 8 and what is the final average?"
)

print(response)

2026-06-09 20:14:15,070 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-09 20:14:17,826 - INFO - Selecting query engine 3: The question specifically asks about Grade 8 students, making the summary for Grade 8 most relevant..
2026-06-09 20:14:18,576 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Betelhem Dawit is the top student in Grade 8 with a final average of 93.25.


### Explanation

```python
build_grade_engine()
```

creates a separate index and query engine for one grade.

```python
QueryEngineTool
```

wraps each query engine and adds a description.

```python
RouterQueryEngine
```

uses the question and the tool descriptions to choose the best query engine.

This is useful when one large index is not the best design and you want more control.

## 22.7 Sub-Question Query Engine

Some questions require information from multiple documents.

Example:

```text
Compare Grade 6 and Grade 8 performance.
```

This question may need multiple smaller questions:

```text
What is Grade 6's class average?
Who is the top student in Grade 6?
Which Grade 6 students need support?
What is Grade 8's class average?
Who is the top student in Grade 8?
Which Grade 8 students need support?
```

The sub-question query engine can break one big question into smaller questions, ask the right tools, and combine the answers.

In [35]:
%%capture
!pip install llama-index-question-gen-openai

In [36]:
query_engine_tools = [
    QueryEngineTool.from_defaults(
        query_engine=grade_5_engine,
        name="grade_5_tool", # Explicit naming helps the router match correctly
        description="Useful for questions about Grade 5 students, scores, attendance, and class summary."
    ),
    QueryEngineTool.from_defaults(
        query_engine=grade_6_engine,
        name="grade_6_tool",
        description="Useful for questions about Grade 6 students, scores, attendance, and class summary."
    ),
    QueryEngineTool.from_defaults(
        query_engine=grade_7_engine,
        name="grade_7_tool",
        description="Useful for questions about Grade 7 students, scores, attendance, and class summary."
    ),
    QueryEngineTool.from_defaults(
        query_engine=grade_8_engine,
        name="grade_8_tool",
        description="Useful for questions about Grade 8 students, scores, attendance, and class summary."
    ),
]


In [37]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.core.response_synthesizers import get_response_synthesizer

# 1. Build the prompt-based question generator using your active settings
question_gen = LLMQuestionGenerator.from_defaults()

# 2. Build the standard response synthesizer
response_synthesizer = get_response_synthesizer()

# 3. Instantiate the engine directly using the correct argument names
sub_question_engine = SubQuestionQueryEngine(
    question_gen=question_gen,                # Changed parameter name from question_generator to question_gen
    response_synthesizer=response_synthesizer,
    query_engine_tools=query_engine_tools,
    verbose=True
)


response = sub_question_engine.query(
    "Compare Grade 6 and Grade 8 performance. Mention top students, class average, strongest subject, and students needing support."
)

print(response)

2026-06-09 20:17:09,931 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Generated 8 sub questions.
[grade_6_tool] Q: Who are the top students in Grade 6?


2026-06-09 20:17:14,814 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[grade_6_tool] A: Isaac Daniel is the top student in Grade 6, with a final average of 89.75.
[grade_6_tool] Q: What is the class average for Grade 6?


2026-06-09 20:17:24,423 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


...

[grade_8_tool] A: The strongest subject in Grade 8 is Science.
[grade_8_tool] Q: Which students in Grade 8 need additional support?


2026-06-09 20:17:43,158 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[grade_8_tool] A: Ermias Kebede needs additional support.


2026-06-09 20:17:45,671 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Grade 6
• Top student: Isaac Daniel (final average 89.75)  
• Class average: 77.2  
• Strongest subject: Science  
• Students needing support: Leul Assefa, Natnael Worku  

Grade 8
• Top student: Betelhem Dawit (final average 93.25)  
• Class average: 84.2  
• Strongest subject: Science  
• Student needing support: Ermias Kebede  

Overall, Grade 8 outperforms Grade 6 in top‐student averages and overall class average, while both grades share Science as their strongest subject.


### Explanation

The sub-question query engine can do this:

```text
complex question
↓
sub-question 1
↓
sub-question 2
↓
sub-question 3
↓
combine answers
↓
final response
```

This is useful when a question spans multiple files or categories.

## 22.8 Advanced Retrieval Evaluation

Earlier, we tested whether the final answer contained an expected keyword.

Now we test the retrieval step.

For each question, we define the expected source file.

Example:

```text
Question:
Who is the top student in Grade 8?

Expected retrieved file:
grade_8_report.txt
```

If the retriever finds the correct source file, the retrieval step is working better.

In [38]:
retrieval_tests = [
    {
        "question": "Who is the top student in Grade 5?",
        "expected_file": "grade_5_report.txt"
    },
    {
        "question": "Which students need support in Grade 6?",
        "expected_file": "grade_6_report.txt"
    },
    {
        "question": "Who is the top student in Grade 7?",
        "expected_file": "grade_7_report.txt"
    },
    {
        "question": "Who is the top student in Grade 8?",
        "expected_file": "grade_8_report.txt"
    },
]

for test in retrieval_tests:
    results = retriever.retrieve(test["question"])

    retrieved_files = [
        result.metadata.get("file_name", "")
        for result in results
    ]

    passed = test["expected_file"] in retrieved_files

    print("Question:", test["question"])
    print("Expected file:", test["expected_file"])
    print("Retrieved files:", retrieved_files)
    print("Passed:", passed)
    print("-" * 80)

Question: Who is the top student in Grade 5?
Expected file: grade_5_report.txt
Retrieved files: ['grade_5_report.txt']
Passed: True
--------------------------------------------------------------------------------
Question: Which students need support in Grade 6?
Expected file: grade_6_report.txt
Retrieved files: ['grade_6_report.txt']
Passed: True
--------------------------------------------------------------------------------
Question: Who is the top student in Grade 7?
Expected file: grade_7_report.txt
Retrieved files: ['grade_7_report.txt']
Passed: True
--------------------------------------------------------------------------------
Question: Who is the top student in Grade 8?
Expected file: grade_8_report.txt
Retrieved files: ['grade_8_report.txt']
Passed: True
--------------------------------------------------------------------------------


## 22.9 Final Advanced RAG Pattern

Now the advanced RAG pipeline looks like this:
text
User question
↓
Query rewriting
↓
Hybrid retrieval
↓
Reranking
↓
Router chooses correct data source
↓
Sub-question engine handles complex questions
↓
LLM generates final answer
↓
Source chunks are shown
↓
Evaluation checks retrieval quality
```

You do not always need every advanced technique.

Use the technique based on the problem:

| Problem | Useful technique |
|---|---|
| Exact names or terms are missed | BM25 / hybrid search |
| Retrieved chunks are noisy | Reranking |
| User question is vague | Query rewriting |
| Many clear data categories exist | Router query engine |
| Question needs multiple documents | Sub-question query engine |
| You are unsure retrieval is correct | Retrieval evaluation |
```

# 23. Final Project: AI Study Assistant Web App

Now we build a complete small web app using Gradio.

The app will:

1. Upload TXT or PDF files
2. Build a LlamaIndex vector index
3. Ask questions
4. Show the answer
5. Show source Grade 8

This is the final project for the crash course.

In [40]:
import shutil
import gradio as gr

app_query_engine = None

def build_index_from_uploads(files):
    """
    Build a LlamaIndex vector index from uploaded TXT or PDF files.

    Parameters:
        files: list of uploaded files from Gradio

    Returns:
        A status message showing whether indexing succeeded.
    """
    global app_query_engine

    if files is None or len(files) == 0:
        return "Please upload at least one TXT or PDF file."

    upload_dir = Path("uploaded_docs")

    if upload_dir.exists():
        shutil.rmtree(upload_dir)

    upload_dir.mkdir(exist_ok=True)

    for file in files:
        source_path = Path(file.name)
        target_path = upload_dir / source_path.name
        shutil.copy(source_path, target_path)

    documents = SimpleDirectoryReader(str(upload_dir)).load_data()

    splitter = SentenceSplitter(
        chunk_size=512,
        chunk_overlap=50
    )

    nodes = splitter.get_nodes_from_documents(documents)

    app_index = VectorStoreIndex(nodes)

    app_query_engine = app_index.as_query_engine(
        similarity_top_k=3,
        text_qa_template=qa_prompt
    )

    return f"Index created successfully. Files: {len(files)} | Chunks: {len(nodes)}"


def ask_question(question):
    """
    Ask a question from the uploaded documents.

    Parameters:
        question: user question as text

    Returns:
        Answer plus source Grade 8.
    """
    global app_query_engine

    if app_query_engine is None:
        return "Please upload files and click Build Index first."

    if question is None or not question.strip():
        return "Please enter a question."

    response = app_query_engine.query(question)

    output = "ANSWER\n"
    output += str(response)
    output += "\n\nSOURCE CHUNKS\n"

    for i, source in enumerate(response.source_nodes, start=1):
        text = source.node.text[:700].replace("\n", " ")
        score = source.score
        metadata = source.node.metadata

        output += f"\nSource {i}\n"
        output += f"Score: {score}\n"
        output += f"Metadata: {metadata}\n"
        output += f"Text: {text}\n"
        output += "-" * 70 + "\n"

    return output


with gr.Blocks() as demo:
    gr.Markdown("# LlamaIndex AI Study Assistant")
    gr.Markdown(
        "Upload TXT/PDF files, build a vector index, ask questions, and inspect source Grade 8."
    )

    files = gr.File(
        label="Upload TXT or PDF files",
        file_count="multiple"
    )

    build_button = gr.Button("Build Index")
    build_status = gr.Textbox(label="Index Status")

    question_box = gr.Textbox(
        label="Ask a question",
        placeholder="Example: Who is the top student in Grade 8?"
    )

    ask_button = gr.Button("Ask")
    answer_box = gr.Textbox(
        label="Answer and Sources",
        lines=22
    )

    build_button.click(
        fn=build_index_from_uploads,
        inputs=files,
        outputs=build_status
    )

    ask_button.click(
        fn=ask_question,
        inputs=question_box,
        outputs=answer_box
    )

demo.launch(share=True)

2026-06-09 20:23:31,765 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-06-09 20:23:31,968 - INFO - HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-06-09 20:23:31,981 - INFO - HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861


2026-06-09 20:23:32,154 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-06-09 20:23:32,602 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://be8d7c4c8ee1580103.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-06-09 20:23:33,955 - INFO - HTTP Request: HEAD https://be8d7c4c8ee1580103.gradio.live "HTTP/1.1 200 OK"


2026-06-09 20:23:34,154 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-06-09 20:24:30,071 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


# 24. Final Summary

You completed a full LlamaIndex crash course from beginner to advanced.

In this notebook, you learned the complete flow:

```text
RAG from scratch
↓
OpenRouter API setup
↓
HuggingFace embeddings
↓
Document loading
↓
Vector indexing
↓
Query engine
↓
Retrieval inspection
↓
Source nodes
↓
Chunking
↓
Custom prompts
↓
PDF question answering
↓
Save and reload index
↓
Chat engine
↓
Simple evaluation
↓
Tools and agents
↓
Advanced retrieval
↓
Final Gradio web app
```

---

## Main beginner concepts

You learned:

```text
What LlamaIndex is
What RAG is
What documents are
What chunks/nodes are
What embeddings are
What a vector index is
What a retriever does
What a query engine does
```

The basic RAG flow is:

```text
User question
↓
Retriever finds relevant chunks
↓
LLM receives context + question
↓
LLM generates answer
```

---

## Main medium concepts

You learned how to improve a basic RAG app using:

```text
custom chunking
chunk overlap
source nodes
custom prompts
PDF loading
index persistence
chat engine
simple evaluation
```

This makes the system clearer, easier to debug, and more useful.

---

## Main advanced concepts

You learned advanced LlamaIndex and RAG techniques:

```text
Hybrid search
BM25 keyword retrieval
Vector retrieval
QueryFusionRetriever
Reranking
Query rewriting
Router Query Engine
Sub-Question Query Engine
Advanced retrieval evaluation
```

The advanced RAG pattern is:

```text
User question
↓
Optional query rewriting
↓
Hybrid retrieval
↓
Optional reranking
↓
Optional routing
↓
Optional sub-question decomposition
↓
LLM answer
↓
Source inspection
↓
Evaluation
```

---

## Final project

The final project is an AI Study Assistant Web App.

It can:

```text
upload TXT/PDF documents
build a LlamaIndex vector index
ask questions from uploaded files
answer using OpenRouter
use HuggingFace embeddings
show source chunks
```

---

## Full system flow

```text
User uploads documents
↓
SimpleDirectoryReader loads the files
↓
SentenceSplitter creates chunks
↓
HuggingFaceEmbedding converts chunks into vectors
↓
VectorStoreIndex stores searchable vectors
↓
Retriever finds relevant chunks
↓
OpenRouter LLM receives context and question
↓
Answer is generated
↓
Source chunks are displayed
```

This is the core pattern behind many document question-answering and RAG applications.